In [ ]:
import pandas as pd
import os

from pathlib import Path
DATA_PATH = str(Path().resolve().parent / "data" / "raw")

seeds = pd.read_csv(f"{DATA_PATH}/MNCAATourneySeeds.csv")
tourney  = pd.read_csv(f"{DATA_PATH}/MNCAATourneyCompactResults.csv")
detailed = pd.read_csv(f"{DATA_PATH}/MRegularSeasonDetailedResults.csv")
massey   = pd.read_csv(f"{DATA_PATH}/MMasseyOrdinals.csv")
sample   = pd.read_csv(f"{DATA_PATH}/SampleSubmissionStage2.csv")

print("=== Seeds ===")
print(seeds.shape)
print(seeds.head(3).to_string())
print("\n=== Tourney Results ===")
print(tourney.shape)
print(tourney.head(3).to_string())
print("\n=== Detailed Results ===")
print(detailed.shape)
print(detailed.head(3).to_string())
print("\n=== Massey Ordinals ===")
print(massey.shape, massey['SystemName'].nunique(), "systems")
print(massey.head(3).to_string())
print("\n=== Sample Submission (Stage 2) ===")
print(sample.shape)
print(sample.head(5).to_string())

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
DATA_PATH = str(Path().resolve().parent / "data" / "raw")

detailed = pd.read_csv(f"{DATA_PATH}/MRegularSeasonDetailedResults.csv")
compact  = pd.read_csv(f"{DATA_PATH}/MRegularSeasonCompactResults.csv")
seeds    = pd.read_csv(f"{DATA_PATH}/MNCAATourneySeeds.csv")
tourney  = pd.read_csv(f"{DATA_PATH}/MNCAATourneyCompactResults.csv")
massey   = pd.read_csv(f"{DATA_PATH}/MMasseyOrdinals.csv")
coaches  = pd.read_csv(f"{DATA_PATH}/MTeamCoaches.csv")
teams    = pd.read_csv(f"{DATA_PATH}/MTeams.csv")

# Sanity checks
print("Detailed seasons:", detailed['Season'].min(), "–", detailed['Season'].max())
print("Compact seasons:", compact['Season'].min(), "–", compact['Season'].max())
print("Tourney seasons:", tourney['Season'].min(), "–", tourney['Season'].max())
print("Seeds seasons:", seeds['Season'].min(), "–", seeds['Season'].max())
print("Massey systems sample:", massey['SystemName'].unique()[:20])
print("Coaches sample:\n", coaches.head(5).to_string())
print("\nAny nulls in detailed?", detailed.isnull().sum().sum())
print("Any nulls in compact?", compact.isnull().sum().sum())

In [ ]:
import pandas as pd

from pathlib import Path
DATA_PATH = str(Path().resolve().parent / "data" / "raw")

massey   = pd.read_csv(f"{DATA_PATH}/MMasseyOrdinals.csv")
teams    = pd.read_csv(f"{DATA_PATH}/MTeams.csv")

# Check coverage of Massey systems by season
for system in ['POM', 'SAG', 'MAS']:
    coverage = (massey[massey['SystemName'] == system]
                .groupby('Season')['TeamID'].count())
    print(f"\n{system} coverage:")
    print(f"  Seasons: {coverage.index.min()} - {coverage.index.max()}")
    print(f"  2024 teams: {coverage.get(2024, 0)}")
    print(f"  2025 teams: {coverage.get(2025, 0)}")
    print(f"  2026 teams: {coverage.get(2026, 0)}")

# Find systems with full coverage through 2026
system_coverage = (massey.groupby('SystemName')['Season']
                   .max()
                   .reset_index()
                   .rename(columns={'Season':'MaxSeason'}))

# Filter to systems that cover 2026 and have been around since at least 2010
system_start = (massey.groupby('SystemName')['Season']
                .min()
                .reset_index()
                .rename(columns={'Season':'MinSeason'}))

coverage_full = (system_coverage
                 .merge(system_start, on='SystemName')
                 .query('MaxSeason == 2026 and MinSeason <= 2010')
                 .sort_values('MinSeason'))

print(f"Systems with coverage 2010-2026: {len(coverage_full)}")
print(coverage_full.to_string(index=False))


# Check NET coverage in the Massey data
net_check = massey[massey['SystemName'].str.contains('NET', case=False, na=False)]
print("NET-related systems:", net_check['SystemName'].unique())
print("NET coverage by season:")
if len(net_check) > 0:
    print(net_check.groupby(['SystemName','Season'])['TeamID'].count().reset_index().to_string())

In [ ]:
# Team lookup!

from pathlib import Path
DATA_PATH = str(Path().resolve().parent / "data" / "raw")

teams    = pd.read_csv(f"{DATA_PATH}/MTeams.csv")

opp_ids = [1112, 1181, 1196, 1276]
print(teams[teams['TeamID'].isin(opp_ids)][['TeamID','TeamName']])